# Two logical qubits: transversal Clifford+T with a *rotation-tracked* Hadamard

This notebook applies **general two-qubit circuits** to two logical qubits encoded in **rotated planar surface codes**, simulated as a single **ITensor MPS**. It keeps the project's conventions — the same $d=3$ rotated-planar encoding, the side-by-side MPS layout, and a *simple* decoder — and focuses on doing every gate **transversally and correctly**:

| logical gate | how we do it here |
|---|---|
| `X`, `Z` | transversal physical Pauli on the logical support |
| `CNOT` | **transversal** bitwise physical CNOT between the two patches |
| `H` | **transversal `H⊗ⁿ` followed by a 90° rotation of the patch** (§3) |
| `S`, `T` | gate teleportation from a magic ancilla patch (§4) |

### The one idea that makes this work

A naïve transversal Hadamard `H⊗ⁿ` on all data qubits *is* a logical $H$, **but it puts the patch into its *dual* code**: every $X$-stabiliser becomes a $Z$-stabiliser and vice-versa. A subsequent transversal CNOT between a dual-code patch and a normal-code patch is **not** a valid logical gate — this is the wall that a *deferred / bookkeeping-only* Hadamard cannot get around.

Following **Chen, Chen, Lu & Pan, *Transversal Logical Clifford gates on rotated surface codes with reconfigurable neutral atom arrays*, Phys. Rev. Lett. (2024)** ([arXiv:2412.01391](https://arxiv.org/abs/2412.01391), PRL `10.1103/m7tq-9v3g`), we **physically rotate the patch by 90°** after the transversal `H⊗ⁿ`. The paper implements the rotation as *horizontal reflection ∘ diagonal reflection* using two sets of 2D acousto-optic deflectors ("a rotation is equivalent to two successive reflections"). The rotation maps the swapped stabilisers **back onto the standard layout**, so the patch is a *bona-fide standard rotated surface code again*. Then a plain transversal CNOT is valid — for **either** orientation of the Hadamard, including `H` on the CNOT **control**.

Because the rotation is a *physical relabelling of the qubit coordinates* (moving atoms), we never carry a "pending H" basis flag: after `H` the logical operators are the standard ones again, so readout, error correction and further gates all stay simple. Everything below is validated against an exact 2-qubit statevector reference in §7.

## §1 — The rotated planar surface code encoding

We use the standard **distance-3 rotated planar code**, unchanged from the earlier notebooks in this project.

- **Data qubits** sit at integer coordinates $(x,y)$, $x,y\in\{0,1,2\}$ — 9 per patch.
- **Plaquette stabilisers** sit at half-integer coordinates. A plaquette at $(x{+}\tfrac12,\,y{+}\tfrac12)$ is a **$Z$-check** when $x{+}y$ is even and an **$X$-check** when $x{+}y$ is odd; boundary checks are weight-2. Each is measured with one ancilla qubit.
- **Logical operators** (in the standard orientation, before any Hadamard): logical $Z$ = $Z$ on the bottom row $y=0$; logical $X$ = $X$ on the left column $x=0$.

**Three patches on one MPS.** Patches 1 and 2 are the two logical data qubits; patch 3 is a recyclable ancilla used for magic-state injection. They are laid **side-by-side** on the MPS — all of patch 1, then patch 2, then patch 3 (data, then $Z$-ancillas, then $X$-ancillas within each patch). This is deliberate: a between-patch transversal CNOT then spans the MPS non-locally, so its true bond-dimension cost is *visible* — the layout we would need when scaling to many more qubits, rather than an interleaving that hides it.

**Experimental preparation.** We prepare each data patch the way an experiment would: reset the data qubits to $|0\rangle$, seed the desired logical state, then **measure the stabilisers** — whose outcomes are *random* — and **feed-forward a Pauli recovery** that flips the lit stabilisers back to $+1$. The one subtlety (§1 code cell) is that the recovery must be a **destabiliser**: a correction with *even* overlap on the logical support, so it carries no logical component and therefore fixes the Pauli frame *without disturbing the logical value*. (A minimum-weight decoder could accidentally apply a logical operator — that is the trap an earlier attempt fell into.) The result is a genuine trivial-frame codeword, which is what lets us read logical operators *exactly* via `joint_expect` (§2) with no per-shot frame ambiguity — even after a Hadamard. Every circuit **starts from $|0\rangle_L\otimes|0\rangle_L$**; any other input is built with leading gates in the circuit (an `H` for $|+\rangle$, an `X` for $|1\rangle$, and so on).

In [1]:
using ITensors, ITensorMPS, LinearAlgebra, Random
Random.seed!(0xB011)
const threshold = 1e-6          # MPS truncation cutoff
const d = 3                     # code distance

# --- physical single-qubit gates not built into ITensors, plus an explicit SWAP ---
ITensors.op(::OpName"T", ::SiteType"S=1/2") = ComplexF64[1 0; 0 exp(im*π/4)]
ITensors.op(::OpName"S", ::SiteType"S=1/2") = ComplexF64[1 0; 0 im]
# a 2-qubit SWAP -- this is how we physically "move an atom" to rotate the patch (§3)
ITensors.op(::OpName"SWAPg", ::SiteType"S=1/2") = ComplexF64[1 0 0 0; 0 0 1 0; 0 1 0 0; 0 0 0 1]

# --- geometry of a single patch: data on the integer grid, checks on half-integers ---
data_coords = vec([(x, y) for x in 0:(d-1), y in 0:(d-1)])
z_aux_local = Tuple{Float64,Float64}[]
x_aux_local = Tuple{Float64,Float64}[]
for x in 0:(d-2), y in 0:(d-2)
    push!((x + y) % 2 == 0 ? z_aux_local : x_aux_local, (x + 0.5, y + 0.5))
end
for y in 1:2:(d-2);  push!(z_aux_local, (-0.5,    y + 0.5)); end   # boundary Z-checks
for y in 0:2:(d-2);  push!(z_aux_local, (d - 0.5, y + 0.5)); end
for x in 0:2:(d-2);  push!(x_aux_local, (x + 0.5, -0.5));    end   # boundary X-checks
for x in 1:2:(d-2);  push!(x_aux_local, (x + 0.5, d - 0.5)); end

# --- side-by-side MPS layout: patch 1, then 2, then 3; data, Z-aux, X-aux within each ---
all_keys = Tuple{Int, Tuple{Float64,Float64}}[]
for p in 1:3
    for q in data_coords;  push!(all_keys, (p, q)); end
    for a in z_aux_local;  push!(all_keys, (p, a)); end
    for a in x_aux_local;  push!(all_keys, (p, a)); end
end
sites   = siteinds("S=1/2", length(all_keys))
site_of = Dict(k => sites[i] for (i, k) in enumerate(all_keys))

data_neighbors_of(ac) =
    [q for q in data_coords if abs(q[1]-ac[1])==0.5 && abs(q[2]-ac[2])==0.5]
P_up(s) = 0.5*op("Id", s) + op("Sz", s)     # projector onto |0>
P_dn(s) = 0.5*op("Id", s) - op("Sz", s)     # projector onto |1>

println("patch: $(length(data_coords)) data, $(length(z_aux_local)) Z-checks, ",
        "$(length(x_aux_local)) X-checks   |   MPS sites (3 patches): $(length(sites))")

patch: 9 data, 4 Z-checks, 4 X-checks   |   MPS sites (3 patches): 51


In [2]:
# ---- projective single-site Z measurement (bit 0 = |0>) ----
function measure_Z!(psi, site; cutoff = threshold)
    sz = real(inner(psi', apply(op("Sz", site), psi; cutoff)))
    if rand() < 0.5 + sz
        psi = apply(P_up(site), psi; cutoff); bit = 0
    else
        psi = apply(P_dn(site), psi; cutoff); bit = 1
    end
    bit, psi / sqrt(real(inner(psi, psi)))
end
reset_aux!(psi, aux; cutoff = threshold) =
    (real(inner(psi', apply(op("Sz", aux), psi; cutoff))) < 0 ? apply(op("X", aux), psi; cutoff) : psi)

# ---- syndrome extraction: measure one Z- or X-plaquette via its ancilla ----
function measure_Z_stab(psi, p, ac; cutoff = threshold)
    aux = site_of[(p, ac)]; nbrs = data_neighbors_of(ac)
    order = length(nbrs) == 4 ? [2,4,1,3] : [1,2]
    for q in nbrs[order]; psi = apply(op("CNOT", site_of[(p,q)], aux), psi; cutoff); end
    measure_Z!(psi, aux; cutoff)
end
function measure_X_stab(psi, p, ac; cutoff = threshold)
    aux = site_of[(p, ac)]; psi = apply(op("H", aux), psi; cutoff)
    nbrs = data_neighbors_of(ac); order = length(nbrs) == 4 ? [2,1,4,3] : [1,2]
    for q in nbrs[order]; psi = apply(op("CNOT", aux, site_of[(p,q)]), psi; cutoff); end
    psi = apply(op("H", aux), psi; cutoff)
    measure_Z!(psi, aux; cutoff)
end
# one full round of both stabiliser types -> (z_syn, x_syn) bit-vectors (RANDOM outcomes)
function project_to_codespace!(psi, p; cutoff = threshold)
    z = Int[]; for ac in z_aux_local
        psi = reset_aux!(psi, site_of[(p,ac)]; cutoff); b, psi = measure_Z_stab(psi,p,ac;cutoff); push!(z,b); end
    x = Int[]; for ac in x_aux_local
        psi = reset_aux!(psi, site_of[(p,ac)]; cutoff); b, psi = measure_X_stab(psi,p,ac;cutoff); push!(x,b); end
    (; z_syn = z, x_syn = x), psi
end

# ---- logical-state seeds ----
const XL_col0    = [(0, y) for y in 0:(d-1)]      # logical-X support (left column x=0)
const ZL_support = [(x, 0) for x in 0:(d-1)]      # logical-Z support (bottom row y=0)
seed_ops(sym) = sym === :zero ? String[] : sym === :one ? ["X"] :
                sym === :plus ? ["H"]   : sym === :minus ? ["H","Z"] :
                sym === :A ? ["H","T"]  : sym === :Y ? ["H","S"] : error("seed $sym")
function reset_patch_data!(psi, p; cutoff = threshold)
    for q in data_coords
        s = site_of[(p, q)]; sz = real(inner(psi', apply(op("Sz", s), psi; cutoff)))
        b = rand() < 0.5 + sz ? 0 : 1
        psi = apply(b == 0 ? P_up(s) : P_dn(s), psi; cutoff); psi = psi / sqrt(real(inner(psi, psi)))
        b == 1 && (psi = apply(op("X", s), psi; cutoff))
    end
    psi
end
println("stabiliser-measurement machinery loaded")

stabiliser-measurement machinery loaded


In [3]:
# =====  experimental logical-state preparation  =====
# Reset -> seed -> MEASURE the stabilisers (random outcomes) -> feed-forward a Pauli
# recovery that flips the lit stabilisers back to +1.  For the recovery not to disturb
# the logical value it must be a *destabiliser*: a correction with EVEN overlap on the
# logical support, so it carries no logical component.  We precompute it as a lookup.

const z_support = Dict(ac => Set(data_neighbors_of(ac)) for ac in z_aux_local)
const x_support = Dict(ac => Set(data_neighbors_of(ac)) for ac in x_aux_local)
# indices of checks flipped by a candidate correction `cset` (set of data coords)
flips(cset, auxs, supp) = sort([i for (i,ac) in enumerate(auxs) if isodd(length(intersect(cset, supp[ac])))])

# syndrome (sorted lit-check indices) -> min-weight correction.
# `avoid`: if given, only keep corrections with EVEN overlap on it (=> no logical content).
function build_lookup(auxs, supp; avoid = nothing)
    coords = collect(data_coords); A = avoid === nothing ? nothing : Set(avoid)
    table = Dict{Vector{Int}, Vector{Tuple{Int,Int}}}()
    for mask in 0:(2^length(coords) - 1)
        e = Set(coords[i] for i in 1:length(coords) if (mask >> (i-1)) & 1 == 1)
        A !== nothing && isodd(length(intersect(e, A))) && continue
        k = flips(e, auxs, supp); w = length(e)
        (!haskey(table,k) || w < length(table[k])) && (table[k] = collect(e))
    end
    table
end
# destabiliser recoveries: Z-syndrome -> X-recovery (commutes with Z_L);  x -> Z-recovery (commutes with X_L)
const XREC = build_lookup(z_aux_local, z_support; avoid = ZL_support)
const ZREC = build_lookup(x_aux_local, x_support; avoid = XL_col0)

# prepare patch p in logical state `sym` the way an experiment would:
function prepare_logical!(psi, p, sym; cutoff = threshold)
    psi = reset_patch_data!(psi, p; cutoff)                          # reset data to |0>
    c = site_of[(p,(0,0))]
    for g in seed_ops(sym); psi = apply(op(g, c), psi; cutoff); end   # seed the logical state
    for q in XL_col0[2:end]; psi = apply(op("H", site_of[(p,q)]), psi; cutoff); end
    (; z_syn, x_syn), psi = project_to_codespace!(psi, p; cutoff)     # MEASURE stabilisers (random)
    zl = sort([i for (i,b) in enumerate(z_syn) if b == 1])
    xl = sort([i for (i,b) in enumerate(x_syn) if b == 1])
    for q in get(XREC, zl, Tuple{Int,Int}[]); psi = apply(op("X", site_of[(p,q)]), psi; cutoff); end  # feed-forward
    for q in get(ZREC, xl, Tuple{Int,Int}[]); psi = apply(op("Z", site_of[(p,q)]), psi; cutoff); end
    psi
end
println("experimental prep ready:  ", length(XREC), " X-recoveries, ", length(ZREC), " Z-recoveries")

experimental prep ready:  16 X-recoveries, 16 Z-recoveries


## §2 — Transversal CNOT, Pauli gates, and exact readout

The **transversal CNOT** between two identically-oriented rotated surface codes is just a **bitwise physical CNOT**: for every data coordinate $q$, apply `CNOT(patch_c[q] → patch_t[q])`. Because the two patches share the same stabiliser layout, this maps the stabiliser group to itself and realises a logical CNOT. (On our side-by-side MPS these physical CNOTs reach right across the chain — the non-locality is real and shows up as bond growth, which is the point of the layout.)

Logical **Paulis** are the physical Paulis on the logical support: $Z_L$ on the bottom row, $X_L$ on the left column.

**Readout.** Because the feed-forward prep (§1) leaves each patch in a **trivial Pauli frame**, we can read a logical operator — or a *joint* two-patch operator like $Z_1 Z_2$ — **exactly** as the expectation $\langle\psi|\,O\,|\psi\rangle$ of the corresponding product of physical Paulis, with `joint_expect`. This is non-collapsing and noise-free, so it is the right tool for verifying entangled logical states (a product of two *separately* collapsed per-patch readouts would destroy the correlation of a Bell state). The trivial frame is essential here: with a *random* frame the sign of $\langle X_L\rangle$ would flip shot-to-shot once a Hadamard mixes the bases — which is exactly why the prep bothers to feed-forward a recovery.

In [4]:
# ---- transversal logical gates ----
apply_ZL!(psi, p; cutoff = threshold) =
    (for q in ZL_support; psi = apply(op("Z", site_of[(p,q)]), psi; cutoff); end; psi)
apply_XL!(psi, p; cutoff = threshold) =
    (for q in XL_col0;    psi = apply(op("X", site_of[(p,q)]), psi; cutoff); end; psi)

function logical_CNOT!(psi, p_ctrl, p_tgt; cutoff = threshold)
    for q in data_coords                                   # bitwise, across the whole patch
        psi = apply(op("CNOT", site_of[(p_ctrl,q)], site_of[(p_tgt,q)]), psi; cutoff)
    end
    psi
end

# ---- exact logical readout ----
# physical (Pauli-type, support) for logical L on a patch in the STANDARD orientation.
logop(L) = L === :Z ? (:Z, ZL_support) : (:X, XL_col0)
# apply the joint operator O = prod over specs of the logical Pauli, to a copy of psi.
# specs :: Vector of (patch, L)   with L in (:Z, :X)
function _apply_joint(psi, specs; cutoff = threshold)
    Opsi = copy(psi)
    for (p, L) in specs
        basis, supp = logop(L)
        for q in supp; Opsi = apply(op(string(basis), site_of[(p,q)]), Opsi; cutoff); end
    end
    Opsi
end
# non-collapsing exact expectation <prod logical Paulis>
joint_expect(psi, specs; cutoff = threshold) = real(inner(psi, _apply_joint(psi, specs; cutoff)))
println("transversal gates + exact readout loaded")

transversal gates + exact readout loaded


## §3 — The transversal Hadamard as `H⊗ⁿ` + a 90° patch rotation

This is the heart of the notebook. In the Heisenberg picture, `H⊗ⁿ` conjugates every stabiliser by swapping $X\leftrightarrow Z$:

$$ Z\text{-plaquette} \;\xrightarrow{\;H^{\otimes n}\;}\; X\text{-operator on the same qubits}, \qquad X\text{-plaquette}\;\to\; Z\text{-operator}. $$

So after `H⊗ⁿ` the checks live at the *wrong* positions — the patch is in its **dual code**. On a *rotated* surface code the fix is a **90° rotation of the whole patch**, which the paper builds from *two reflections* (horizontal, then diagonal — Fig. 1 of arXiv:2412.01391). A 90° rotation maps $Z$-plaquette positions onto $X$-plaquette positions (they interleave on a checkerboard), so it carries each swapped operator back onto a position where that *type* of check belongs → the layout is standard again.

Concretely, the rotation acts on data coordinates as

$$ \sigma:\;(x,y)\;\longmapsto\;(y,\;d{-}1{-}x). $$

We realise it **physically** — as the paper does, by *moving the atoms* — with a network of SWAP gates that permutes the data-qubit states according to $\sigma$ (`logical_H_rot!` below). After `H⊗ⁿ` followed by this permutation:

- the patch is a **standard rotated surface code** again (trivial syndrome, verified below), and
- its logical operators are the standard ones again ($Z_L$ = bottom row, $X_L$ = left column), now holding the **Hadamard-transformed** logical value.

That is the whole trick: because we *physically* rotate rather than merely relabel in bookkeeping, there is **no residual basis flag**. A following transversal CNOT sees two ordinary standard-orientation patches and is a valid logical gate — for `H` on the control *or* the target. (`σ` has order 4: `σ²` is a 180° rotation = logical identity up to a relabel, `σ⁴ = 1`, matching `H² = I`.)

In [6]:
# the 90-degree rotation on integer data coordinates
sigma(q) = (q[2], (d - 1) - q[1])

# decompose sigma into cycles, then into adjacent transpositions (SWAP gates)
function sigma_cycles()
    seen = Set{Tuple{Int,Int}}(); cycles = Vector{Vector{Tuple{Int,Int}}}()
    for q0 in data_coords
        q0 in seen && continue
        cyc = Tuple{Int,Int}[]; q = q0
        while !(q in seen); push!(cyc, q); push!(seen, q); q = sigma(q); end
        length(cyc) > 1 && push!(cycles, cyc)
    end
    cycles
end
const SIGMA_CYCLES = sigma_cycles()

# logical H on patch p:  transversal H on every data qubit, then physically move the
# atoms by sigma (SWAP network) -- i.e. rotate the patch back to the standard layout.
function logical_H_rot!(psi, p; cutoff = threshold)
    for q in data_coords
        psi = apply(op("H", site_of[(p, q)]), psi; cutoff)          # H^{\otimes n}
    end
    for cyc in SIGMA_CYCLES, i in 1:(length(cyc) - 1)               # permute states by sigma
        psi = apply(op("SWAPg", site_of[(p, cyc[i])], site_of[(p, cyc[i+1])]), psi; cutoff)
    end
    psi
end

# --- demonstrate: H|0> = |+>, still a valid codeword (trivial syndrome) ---
psi = MPS(sites, "Up")
psi = prepare_logical!(psi, 1, :zero)
psi = logical_H_rot!(psi, 1)
syn, _ = project_to_codespace!(copy(psi), 1)
println("sigma sends the data qubits in cycles: ", SIGMA_CYCLES)
println("after H on |0>_L :  <Z_L> = ", round(joint_expect(psi, [(1,:Z)]), digits=4),
        "   <X_L> = ", round(joint_expect(psi, [(1,:X)]), digits=4), "   (=> |+>_L)")
println("syndrome after H (all zero => still a standard codeword):  z=", syn.z_syn, "  x=", syn.x_syn)

sigma sends the data qubits in cycles: [[(0, 0), (0, 2), (2, 2), (2, 0)], [(1, 0), (0, 1), (1, 2), (2, 1)]]
after H on |0>_L :  <Z_L> = -0.0   <X_L> = 1.0   (=> |+>_L)
syndrome after H (all zero => still a standard codeword):  z=[0, 0, 0, 0]  x=[0, 0, 0, 0]


## §4 — Non-Clifford gates: `S` and `T` by teleportation

`S` and `T` are not transversal on the surface code, so we inject them from **magic states** on the ancilla (patch 3):

- $|Y\rangle = S|+\rangle$ teleports an $S$; $|A\rangle = T|+\rangle$ teleports a $T$.
- The gadget is a transversal CNOT from the data patch onto the freshly-injected ancilla, a destructive logical-$Z$ readout of the ancilla, and a Pauli byproduct. A $T$'s byproduct is an $S$, applied recursively via the $S$-gadget.

Because a Hadamard here is *physically applied and the patch rotated back to standard*, `S`/`T` compose with `H` with **no special handling** — after an `H` the patch is an ordinary standard-orientation code, so `teleport_S!`/`teleport_T!` act exactly as they would with no prior Hadamard. (Here the magic states are injected perfectly; distillation is out of scope.)

In [7]:
# The magic ancilla (patch 3) must be prepared in a CLEAN (trivial) Pauli frame -- exactly
# like the data patches (§1), with the destabiliser feed-forward -- NOT with a leftover
# random frame.  Why it matters: the teleportation gadget couples the ancilla to the data
# patch with a *transversal* CNOT, and a transversal CNOT spreads the ancilla's stabiliser
# frame onto the data patch (Z_anc -> Z_data.Z_anc under conjugation).  If the ancilla
# carried a random frame, that randomness would land on the data patch; the interleaved
# decoder (§5) -- which decodes syndromes against the trivial +1 frame -- would then read
# it as a spurious error and "correct" it, occasionally applying a *logical* (flipping
# <X_L>, so <X1X2> jumps to -0.707 on the magic Bell).  So we reuse prepare_logical!.

# destructive logical-Z of a whole patch (used to read out & discard the ancilla)
function measure_patch_Z!(psi, p; cutoff = threshold)
    par = 0
    for q in data_coords
        s = site_of[(p,q)]; sz = real(inner(psi', apply(op("Sz", s), psi; cutoff)))
        b = rand() < 0.5 + sz ? 0 : 1
        psi = apply(b == 0 ? P_up(s) : P_dn(s), psi; cutoff); psi = psi / sqrt(real(inner(psi,psi)))
        q in ZL_support && (par ⊻= b)
    end
    par, psi
end

# S from |Y>=S|+>, T from |A>=T|+>  (T's byproduct S applied recursively).
# The ancilla is prepared in a trivial frame (prepare_logical!) so the gadget's transversal
# CNOT does not leak a random frame onto the data patch -- see the note above.
function teleport_S!(psi, p; cutoff = threshold)
    psi = prepare_logical!(psi, 3, :Y; cutoff)
    psi = logical_CNOT!(psi, p, 3; cutoff)
    m, psi = measure_patch_Z!(psi, 3; cutoff)
    m == 1 && (psi = apply_ZL!(psi, p; cutoff)); psi
end
function teleport_T!(psi, p; cutoff = threshold)
    psi = prepare_logical!(psi, 3, :A; cutoff)
    psi = logical_CNOT!(psi, p, 3; cutoff)
    m, psi = measure_patch_Z!(psi, 3; cutoff)
    m == 1 && (psi = teleport_S!(psi, p; cutoff)); psi
end
println("magic-state teleportation gadgets loaded")

magic-state teleportation gadgets loaded


## §5 — The decoder (minimum-weight perfect matching)

This is the **MWPM decoder** used throughout the project. For each stabiliser type we build a matching graph whose nodes are the checks (over `R` rounds, with a single boundary node), with unit-weight edges between checks that share a data qubit (spatial), between the same check in adjacent rounds (time), and from boundary checks to the boundary node. `build_graph_generic` runs Floyd–Warshall for all-pairs shortest paths; `mwpm` finds the minimum-weight pairing of the lit **detectors** (here, single-round syndrome changes) to each other or to the boundary; `decode_patch` walks each matched path and collects the data qubits it crosses — the correction.

`decode_correct!` measures one round of both syndromes, decodes each type, and applies the correction as **physical Paulis** ($X$ from the $Z$-graph, $Z$ from the $X$-graph). Because the prep leaves a trivial frame, the corrected state is a genuine codeword again and the exact readout of §2 reports the recovered logical value directly.

Note this is distinct from the **prep recovery** in §1: that one is deliberately constrained to *avoid* logical operators (it fixes the frame of a fresh codeword, which carries no coset information). The decoder here must *not* avoid them — a real error genuinely can be a logical, and minimum weight is the honest guess. At $d=3$ it corrects any single-qubit error; it is single-round for now (no measurement-error / multi-round handling).

In [8]:
const Nz_per_patch = length(z_aux_local)

# ---- Z-stabiliser matching geometry (edges/boundary-edges <-> data qubits) ----
z_aux_containing(q) = [i for (i,a) in enumerate(z_aux_local) if abs(a[1]-q[1])==0.5 && abs(a[2]-q[2])==0.5]
edge_data_local  = Dict{Tuple{Int,Int}, Tuple{Int,Int}}()
bedge_data_local = Dict{Int, Tuple{Int,Int}}()
for q in data_coords
    zs = z_aux_containing(q)
    if length(zs) == 2; a,b = minmax(zs[1],zs[2]); edge_data_local[(a,b)] = q
    elseif length(zs) == 1; bedge_data_local[zs[1]] = q end
end

# ---- spacetime matching graph for R rounds (all-pairs shortest paths) ----
function build_graph_generic(R::Int, Nstab::Int, edge_data, bedge_data)
    Ntot = R*Nstab + 1; BND = Ntot
    node_id(i,r) = (r-1)*Nstab + i
    decode_id(u) = (mod1(u,Nstab), div(u-1,Nstab) + 1)
    dist = fill(Inf, Ntot, Ntot); nxt = fill(-1, Ntot, Ntot)
    for i in 1:Ntot; dist[i,i] = 0.0; nxt[i,i] = i; end
    add!(u,v) = (dist[u,v]=1.0; dist[v,u]=1.0; nxt[u,v]=v; nxt[v,u]=u)
    for r in 1:R, ((i,j),_) in edge_data; add!(node_id(i,r), node_id(j,r)); end   # spatial
    for r in 1:(R-1), i in 1:Nstab; add!(node_id(i,r), node_id(i,r+1)); end        # time
    for r in 1:R, (i,_) in bedge_data; add!(node_id(i,r), BND); end                # boundary
    for k in 1:Ntot, i in 1:Ntot, j in 1:Ntot                                      # Floyd-Warshall
        if dist[i,k] + dist[k,j] < dist[i,j]; dist[i,j] = dist[i,k] + dist[k,j]; nxt[i,j] = nxt[i,k]; end
    end
    function edge_kind(u,v)
        u,v = minmax(u,v)
        v == BND && return (:boundary, bedge_data[decode_id(u)[1]])
        iu,ru = decode_id(u); iv,rv = decode_id(v)
        ru == rv ? (:spatial, edge_data[minmax(iu,iv)]) : (:time, (iu, min(ru,rv)))
    end
    function path(a,b)
        nxt[a,b] == -1 && return Int[]
        p = [a]; while a != b; a = nxt[a,b]; push!(p,a); end; p
    end
    (; dist, nxt, BND, node_id, decode_id, edge_kind, path, R)
end
build_graph(R) = build_graph_generic(R, Nz_per_patch, edge_data_local, bedge_data_local)

# ---- minimum-weight perfect matching (exact, tiny; defects + one boundary each) ----
function mwpm(defects::Vector{Int}, dist::Matrix{Float64}, BND::Int)
    isempty(defects) && return Tuple{Int,Int}[], 0.0
    nd = length(defects); nodes = vcat(defects, fill(BND, nd)); N = length(nodes)
    W = zeros(N,N)
    for i in 1:N, j in (i+1):N
        W[i,j] = (nodes[i]==BND && nodes[j]==BND) ? 0.0 : dist[nodes[i], nodes[j]]; W[j,i] = W[i,j]
    end
    function rec(rem)
        isempty(rem) && return 0.0, Tuple{Int,Int}[]
        first = rem[1]; bc = Inf; bm = Tuple{Int,Int}[]
        for k in 2:length(rem)
            c = W[first, rem[k]]; isfinite(c) || continue
            sc, sm = rec(vcat(rem[2:k-1], rem[k+1:end]))
            c + sc < bc && (bc = c + sc; bm = vcat([(first, rem[k])], sm))
        end
        bc, bm
    end
    cost, m_idx = rec(collect(1:N))
    pairs = [(nodes[i], nodes[j]) for (i,j) in m_idx if !(nodes[i]==BND && nodes[j]==BND)]
    pairs, cost
end

# ---- decode a syndrome history into the set of data qubits to correct ----
function decode_patch(raw_history::Vector{Vector{Int}}, initial_syn::Vector{Int}, g)
    prev = initial_syn; lit = Int[]
    for r in 1:g.R                                             # detectors = syndrome CHANGES
        d = raw_history[r] .⊻ prev
        for (i,b) in enumerate(d); b == 1 && push!(lit, g.node_id(i,r)); end
        prev = raw_history[r]
    end
    pairs, cost = mwpm(lit, g.dist, g.BND)
    qs = Tuple{Int,Int}[]
    for (a,b) in pairs
        p = g.path(a,b)
        for k in 1:(length(p)-1)
            kind = g.edge_kind(p[k], p[k+1])
            kind[1] === :time || push!(qs, kind[2])           # spatial/boundary edge -> a data qubit
        end
    end
    cnt = Dict{Tuple{Int,Int},Int}(); for q in qs; cnt[q] = get(cnt,q,0) + 1; end
    (; frame = [q for (q,c) in cnt if isodd(c)])              # qubits to flip (odd multiplicity)
end

# ---- X-stabiliser graph (for Z errors), mirror of the Z-graph above ----
x_aux_containing(q) = [i for (i,a) in enumerate(x_aux_local) if abs(a[1]-q[1])==0.5 && abs(a[2]-q[2])==0.5]
const Nx_per_patch = length(x_aux_local)
xedge_data_local  = Dict{Tuple{Int,Int}, Tuple{Int,Int}}()
xbedge_data_local = Dict{Int, Tuple{Int,Int}}()
for q in data_coords
    xs = x_aux_containing(q)
    if length(xs) == 2; a,b = minmax(xs[1],xs[2]); xedge_data_local[(a,b)] = q
    elseif length(xs) == 1; xbedge_data_local[xs[1]] = q end
end
build_graph_X(R) = build_graph_generic(R, Nx_per_patch, xedge_data_local, xbedge_data_local)

const GZ = build_graph(1)      # matches Z-syndromes -> X-corrections;  R = 1 (single round)
const GX = build_graph_X(1)    # matches X-syndromes -> Z-corrections

# ---- one round of both raw syndromes, then MWPM-decode and apply physical corrections ----
function measure_raw_syndrome(psi, p; cutoff = threshold)
    z = Int[]; for ac in z_aux_local
        psi = reset_aux!(psi, site_of[(p,ac)]; cutoff); b,psi = measure_Z_stab(psi,p,ac;cutoff); push!(z,b); end
    x = Int[]; for ac in x_aux_local
        psi = reset_aux!(psi, site_of[(p,ac)]; cutoff); b,psi = measure_X_stab(psi,p,ac;cutoff); push!(x,b); end
    z, x, psi
end
function decode_correct!(psi, p; cutoff = threshold)
    z, x, psi = measure_raw_syndrome(psi, p; cutoff)
    for q in decode_patch([z], zeros(Int, Nz_per_patch), GZ).frame; psi = apply(op("X", site_of[(p,q)]), psi; cutoff); end
    for q in decode_patch([x], zeros(Int, Nx_per_patch), GX).frame; psi = apply(op("Z", site_of[(p,q)]), psi; cutoff); end
    psi
end
println("MWPM decoder built (Z-graph: ", GZ.BND-1, " nodes, X-graph: ", GX.BND-1, " nodes)")

MWPM decoder built (Z-graph: 4 nodes, X-graph: 4 nodes)


## §6 — The circuit runner

A circuit is a list of gate tuples on patches 1 and 2. Both patches **always start in $|0\rangle_L$** (experimentally prepared, §1); you build any other input with **leading gates**:

Gate symbols: `:X, :Z, :H, :S, :T` (single-qubit, on patch `p`) and `:CNOT` (control, target). `run_circuit` returns the MPS; read out with `corr(psi, :Z, :Z)` etc. Optional arguments let you **inject physical errors** at chosen points and turn on **interleaved error correction** (`ec=true` decodes both data patches after every gate).

In [9]:
# dispatch a single logical gate onto the MPS
function apply_logical!(psi, gate; cutoff = threshold)
    g = gate[1]
    if     g === :X;    return apply_XL!(psi, gate[2]; cutoff)
    elseif g === :Z;    return apply_ZL!(psi, gate[2]; cutoff)
    elseif g === :H;    return logical_H_rot!(psi, gate[2]; cutoff)     # H^n + 90-deg rotation
    elseif g === :S;    return teleport_S!(psi, gate[2]; cutoff)
    elseif g === :T;    return teleport_T!(psi, gate[2]; cutoff)
    elseif g === :CNOT; return logical_CNOT!(psi, gate[2], gate[3]; cutoff)
    else; error("unknown gate $g"); end
end

# Run a whole circuit. Both data patches ALWAYS start experimentally prepared in |0>_L
# -- build any other input with leading gates (H for |+>, X for |1>, H;T for a magic
# control, ...). `errors` = list of (after_gate_index, patch, "X"/"Z", coord) injected
# after that gate; `ec` decodes both data patches after every gate.
function run_circuit(circuit; ec = false, errors = [], cutoff = threshold)
    psi = MPS(sites, "Up")
    psi = prepare_logical!(psi, 1, :zero; cutoff)
    psi = prepare_logical!(psi, 2, :zero; cutoff)
    for (k, gate) in enumerate(circuit)
        psi = apply_logical!(psi, gate; cutoff)
        for (kk, p, P, q) in errors
            kk == k && (psi = apply(op(P, site_of[(p, q)]), psi; cutoff))
        end
        if ec
            psi = decode_correct!(psi, 1; cutoff)
            psi = decode_correct!(psi, 2; cutoff)
        end
    end
    psi
end
corr(psi, b1, b2) = joint_expect(psi, [(1, b1), (2, b2)])
println("runner ready")

runner ready


## §7 — Validation against an exact 2-qubit reference

We check the encoded runner against a plain $4\times4$ statevector simulator (both qubits starting in $|0\rangle$).

- **`H1; CNOT`** — Hadamard on the **control**, then CNOT: a clean Bell pair. *This is exactly the composition a deferred / transversal-CZ Hadamard cannot do* — here it just works, because the patch was physically rotated back to a standard code.
- **`H2; CNOT`** — Hadamard on the target: a product state, no entanglement.
- **`T1; CNOT` vs `H1; T1; CNOT`** — the first gives no magic (`T` on `|0⟩` is trivial); the second is the magic Bell with `⟨X₁X₂⟩ = cos(π/4) = 0.707`.
- Hadamards on both sides of a CNOT, `HH = I`, and an `S`-based Clifford.

Every value is read exactly with `joint_expect`; the feed-forward prep leaves a trivial frame, so a single run per circuit is exact (no shot averaging needed, even though prep and teleportation both involve random measurement outcomes).

In [10]:
# ---- exact 2-qubit statevector reference (both qubits start in |0>) ----
const _I = ComplexF64[1 0; 0 1];   const _X = ComplexF64[0 1; 1 0]
const _Z = ComplexF64[1 0; 0 -1];  const _H = ComplexF64[1 1; 1 -1]/sqrt(2)
const _S = ComplexF64[1 0; 0 im];  const _T = ComplexF64[1 0; 0 exp(im*π/4)]
_1q(g, p) = p == 1 ? kron(g, _I) : kron(_I, g)
const _CN12 = ComplexF64[1 0 0 0; 0 1 0 0; 0 0 0 1; 0 0 1 0]
function ref_state(circuit)
    ψ = ComplexF64[1, 0, 0, 0]                         # |00>
    for gate in circuit
        g = gate[1]
        M = g === :X ? _1q(_X,gate[2]) : g === :Z ? _1q(_Z,gate[2]) : g === :H ? _1q(_H,gate[2]) :
            g === :S ? _1q(_S,gate[2]) : g === :T ? _1q(_T,gate[2]) : g === :CNOT ? _CN12 : error()
        ψ = M * ψ
    end
    ψ
end
_P = Dict(:X => _X, :Z => _Z)
ref_corr(ψ, b1, b2) = real(ψ' * kron(_P[b1], _P[b2]) * ψ)

# ---- demos: every input state is built with leading gates from |00> ----
demos = [
    ("H1;CNOT        (Bell, H on CONTROL)",     [(:H,1),(:CNOT,1,2)]),
    ("H2;CNOT        (product)",                [(:H,2),(:CNOT,1,2)]),
    ("H1;CNOT;H1     (H after CNOT)",           [(:H,1),(:CNOT,1,2),(:H,1)]),
    ("H1;H1;CNOT     (HH = I)",                 [(:H,1),(:H,1),(:CNOT,1,2)]),
    ("H1;CNOT;H1;H2  (H both sides)",           [(:H,1),(:CNOT,1,2),(:H,1),(:H,2)]),
    ("T1;CNOT        (no leading H: no magic)", [(:T,1),(:CNOT,1,2)]),
    ("H1;T1;CNOT     (magic Bell)",             [(:H,1),(:T,1),(:CNOT,1,2)]),
    ("H1;S1;H1;CNOT",                           [(:H,1),(:S,1),(:H,1),(:CNOT,1,2)]),
    ("H1;H2;T1;T2;CNOT",                           [(:H,1),(:H,2),(:T,1),(:T,2),(:CNOT,1,2)]),
]
println(rpad("circuit",40), rpad("ZZ  sim / ref",20), rpad("XX  sim / ref",20), "ok")
println("-"^84)
for (name, circ) in demos
    ψ = ref_state(circ); rZZ = ref_corr(ψ,:Z,:Z); rXX = ref_corr(ψ,:X,:X)
    psi = run_circuit(circ); zz = corr(psi,:Z,:Z); xx = corr(psi,:X,:X)
    ok = abs(zz-rZZ) < 0.02 && abs(xx-rXX) < 0.02
    println(rpad(name,40),
        rpad("$(round(zz,digits=3)) / $(round(rZZ,digits=3))",20),
        rpad("$(round(xx,digits=3)) / $(round(rXX,digits=3))",20), ok ? "OK" : "MISMATCH")
end

# ---- error-correction demo: inject a Y=X.Z error on patch 1 mid-circuit, correct it ----
# `errors` are keyed by GATE INDEX, so the injection point must track each circuit's length:
# in H1;CNOT the CNOT is gate 2, but in H1;T1;CNOT the CNOT is gate 3 (T is gate 2).
println("\nEC demo -- H1;CNOT (Bell), inject Y on patch-1 qubit (0,0) after the CNOT (gate 2):")
circ = [(:H,1),(:CNOT,1,2)]
bad  = run_circuit(circ; errors = [(2,1,"X",(0,0)), (2,1,"Z",(0,0))], ec = false)
good = run_circuit(circ; errors = [(2,1,"X",(0,0)), (2,1,"Z",(0,0))], ec = true)
println("  uncorrected:  ZZ=", round(corr(bad,:Z,:Z),digits=3),  "  XX=", round(corr(bad,:X,:X),digits=3))
println("  corrected:    ZZ=", round(corr(good,:Z,:Z),digits=3), "  XX=", round(corr(good,:X,:X),digits=3),
        "   (Bell restored)")

# ---- same, on the magic Bell:  here the CNOT is gate 3, so inject at index 3 ----
println("\nEC demo -- H1;T1;CNOT (Magic Bell), inject Y on patch-1 qubit (0,0) after the CNOT (gate 3):")
circ = [(:H,1),(:T,1),(:CNOT,1,2)]
bad  = run_circuit(circ; errors = [(3,1,"X",(0,0)), (3,1,"Z",(0,0))], ec = false)
good = run_circuit(circ; errors = [(3,1,"X",(0,0)), (3,1,"Z",(0,0))], ec = true)
println("  uncorrected:  ZZ=", round(corr(bad,:Z,:Z),digits=3),  "  XX=", round(corr(bad,:X,:X),digits=3))
println("  corrected:    ZZ=", round(corr(good,:Z,:Z),digits=3), "  XX=", round(corr(good,:X,:X),digits=3),
        "   (Magic Bell restored:  ZZ=1, XX=+0.707)")

circuit                                 ZZ  sim / ref       XX  sim / ref       ok
------------------------------------------------------------------------------------
H1;CNOT        (Bell, H on CONTROL)     1.0 / 1.0           1.0 / 1.0           OK
H2;CNOT        (product)                0.0 / -0.0          0.0 / 0.0           OK
H1;CNOT;H1     (H after CNOT)           0.0 / 0.0           -0.0 / -0.0         OK
H1;H1;CNOT     (HH = I)                 1.0 / 1.0           0.0 / -0.0          OK
H1;CNOT;H1;H2  (H both sides)           1.0 / 1.0           1.0 / 1.0           OK
T1;CNOT        (no leading H: no magic) 1.0 / 1.0           0.0 / 0.0           OK
H1;T1;CNOT     (magic Bell)             1.0 / 1.0           0.707 / 0.707       OK
H1;S1;H1;CNOT                           1.0 / 1.0           0.0 / -0.0          OK
H1;H2;T1;T2;CNOT                        -0.0 / -0.0         0.707 / 0.707       OK

EC demo -- H1;CNOT (Bell), inject Y on patch-1 qubit (0,0) after the CNOT (gate 2):


## §7b — Optional: shot-based measurement outcomes & counts

The `joint_expect` readout above is the exact, noise-free expectation value — the cleanest way to see the $ZZ$/$XX$ correlations. But an experiment only ever gets **shots**: destructive single-run measurements of each logical qubit, tallied into counts. The code below does exactly that — for each shot it re-runs prep + circuit and measures both logical qubits (on the *same* collapsing state, so joint correlations survive) in a chosen basis — and reports the outcome histogram and the sampled correlation, so you can watch the statistics approach the exact value as the shot count grows.

This is **off by default** because it is slow (each shot repeats the whole prep + circuit + readout, ~1–2 s per shot). Set `SHOW_COUNTS = true` and pick `NSHOTS` in the next cell to run it. For a stabiliser state (e.g. a Bell pair read in its natural basis) the counts are a clean 50/50 with perfect correlation even at modest shot counts; for a magic state ($\langle XX\rangle=0.707$) you will see the sampled value fluctuate around $0.707$ with a spread $\sim 1/\sqrt{\text{shots}}$.

In [ ]:
# destructive logical measurement of one patch in basis :Z or :X (bit = parity over support).
function measure_logical!(psi, p, basis; cutoff = threshold)
    supp = basis === :Z ? ZL_support : XL_col0
    basis === :X && (for q in data_coords; psi = apply(op("H", site_of[(p,q)]), psi; cutoff); end)
    par = 0
    for q in data_coords
        s = site_of[(p,q)]; sz = real(inner(psi', apply(op("Sz", s), psi; cutoff)))
        b = rand() < 0.5 + sz ? 0 : 1
        psi = apply(b == 0 ? P_up(s) : P_dn(s), psi; cutoff); psi = psi / sqrt(real(inner(psi, psi)))
        q in supp && (par ⊻= b)
    end
    par, psi
end

# run `shots` shots; measure BOTH logical qubits on the same collapsing state each shot.
# returns (counts over the 4 two-bit outcomes, sampled correlation <b1 b2>).
function sample_counts(circuit; basis = (:Z, :Z), shots = 100, ec = false, cutoff = threshold)
    counts = Dict((0,0)=>0, (0,1)=>0, (1,0)=>0, (1,1)=>0)
    for _ in 1:shots
        psi = run_circuit(circuit; ec, cutoff)
        b1, psi = measure_logical!(psi, 1, basis[1]; cutoff)
        b2, psi = measure_logical!(psi, 2, basis[2]; cutoff)
        counts[(b1,b2)] += 1
    end
    ev = sum((1 - 2a) * (1 - 2b) * n for ((a,b), n) in counts) / shots
    counts, ev
end

# ---- toggle the (slow) shot-sampling demo here ----
SHOW_COUNTS = false      # set true to run
NSHOTS      = 100
if SHOW_COUNTS
    cases = [("H1;CNOT   (Bell)",     [(:H,1),(:CNOT,1,2)],        (:Z,:Z), 1.0),
             ("H1;CNOT   (Bell)",     [(:H,1),(:CNOT,1,2)],        (:X,:X), 1.0),
             ("H1;T1;CNOT (magic)",   [(:H,1),(:T,1),(:CNOT,1,2)], (:X,:X), 0.707)]
    for (name, circ, basis, exact) in cases
        c, ev = sample_counts(circ; basis = basis, shots = NSHOTS)
        println("\n$name  measured in $(basis[1])$(basis[2])  ($NSHOTS shots):")
        for k in [(0,0),(0,1),(1,0),(1,1)]
            println("   |$(k[1])$(k[2])>: ", rpad(c[k], 5), " (", round(100*c[k]/NSHOTS, digits=1), "%)")
        end
        println("   sampled <", basis[1], basis[2], "> = ", round(ev, digits=3), "   (exact ", exact, ")")
    end
else
    println("SHOW_COUNTS = false -- set it to true (and pick NSHOTS) to run the slower shot-sampling demo.")
end